## Assignment 2: MongoDB

This assignment is based on content discussed in the Introduction to MongoDB module.

## Learning outcomes 

The purpose of this assignment is for learners to be able to:
- Familarize with JSON document syntax
- Understand basic MongoDB CRUD operations
- Understand MongoDB data pipelines to run aggregate queries

In this assignment, you will make use of the sample data provided.  

This dataset has 3 collections: Employee, Workplace and Address.  You will import this data into your local MongoDB database.

Required imports for this project are given below. Make sure you have all libraries required for this project installed. You may use conda or pip based on your set up.

## Setup Notes

**Please note** These instructions are duplicated here for your reference. You would have encountered them in a previous module. It is expected that you have already taken the steps to set up and run MongoDB.

We will be using MongoDB Community Edition.  The MongoDB database <b>MUST</b> be installed and running locally before continuing with this notebook.  We will need to install two packages using the Anaconda package manager:

1. [Mongodb](https://www.mongodb.com/) - this package contains the mongodb database 
2. [PyMongo](https://www.mongodb.com/docs/drivers/pymongo/) - this package contains the python driver that will allow us to communicate with the mongodb database.

#### Install

1. Open a command line terminal and execute the following command to install mongodb.
```console
conda install -c anaconda mongodb
```
2. Open a command line terminal and execute the following command to install pymongo packge.
```console
conda install -c anaconda pymongo
```

#### Run Mongodb in Windows
1. MongoDB requires a data directory to store all data. MongoDB’s default data directory path is \data\db. Create this folder using the following commands from a Command Prompt:
```console
md \data\db
```

2. To start MongoDB, run mongod.exe. For example, from the Command Prompt:
```console
"C:\Program Files\MongoDB\Server\3.2\bin\mongod.exe"
```

#### Run Mongodb in Mac
1. MongoDB requires a data directory to store all data. MongoDB’s default data directory path is /data/db. Create this folder using the following commands from a Command Prompt.  Note that we run the command as a super user using the "sudo" command:
```console
sudo mkdir /data/db/
```

2. To start MongoDB, run mongod.exe. For example, from the Command Promp.  Note that we run the command as a super user using the "sudo" command:
```console
sudo mongod --config /usr/local/etc/mongod.conf
```

In [85]:
!pip install pymongo

In [76]:
#required imports
import os
import json
import datetime
import pymongo
import pprint
import pandas as pd
import numpy as np
from pymongo import MongoClient
print('Mongo version', pymongo.__version__)

Mongo version 4.16.0


We first need to connect to our locally running MongoDB database (<b>Make sure your database is running on your machine</b>). We will use the MongoClient to connect to a local 'test' database that is running on port 27017 (this is the default port).

In [77]:
client = MongoClient('localhost', 27017)
db = client.assignment1

After installing necessary modules proceed to import the data into your database.

In [78]:
import json

# Delete old collections
db.workplace.drop()
db.address.drop()
db.employee.drop()

# Import files
with open('Employee.json') as f:
    db.employee.insert_many(json.load(f))

with open('Workplace.json') as f:
    db.workplace.insert_many(json.load(f))

with open('Address.json') as f:
    db.address.insert_many(json.load(f))

print("Data imported successfully!")

Data imported successfully!


#### Question 1 (10 Marks)

The address collection contains employee from different ages and interests.  Perform a simple query to list all employees that are less than or equal to 50 and like Cooking.

__NOTE:__ the following shows the structure of an Employee document that will help you construct the query.

In [79]:
# --- Question 1 Implementation ---

# Define the query criteria
# age <= 50 and 'Cooking' exists in the interests array
query = {
    "age": {"$lte": 50},
    "interests": "Cooking"
}

# Execute the search on the employee collection
results = db.employee.find(query)

# Convert the cursor to a list and then to a DataFrame
import pandas as pd
df_results = pd.DataFrame(list(results))

print(f"Found {len(df_results)} employees matching the criteria.")

# Display the first few results with relevant columns
display(df_results[['firstname', 'lastname', 'age', 'interests']].head(10))

Found 16 employees matching the criteria.


,firstname,lastname,age,interests
0,Emilie,Woods,40,"[Bowling, Cooking, Golf, Swimming]"
1,Thomas,Patterson,18,"[Cooking, Cricket, Tennis, Swimming, Fishing]"
2,Sophia,Flores,22,"[Hiking, Soccer, Bowling, Rubgy, Cooking, Danc..."
3,Ollie,Barnett,25,"[Cooking, Bowling, Dancing]"
4,James,Wilkins,27,"[Rubgy, Tennis, Cricket, Cooking]"
5,Aaron,Carr,25,[Cooking]
6,Alta,Sharp,34,"[Cricket, Cycling, Rubgy, Golf, Cooking, Dancing]"
7,Delia,Douglas,36,"[Cricket, Cooking, Hiking, Dancing, Tennis]"
8,Dominic,Wade,48,"[Cycling, Dancing, Cooking]"
9,Myrtle,Little,36,"[Cooking, Cycling, Hiking, Rubgy, Bowling, Dan..."


In [80]:
pprint.pprint(client.assignment1.employee.find_one())

{'_id': '9f39da36-82cc-4353-ab90-d616105fa7c1',
 'address_id': 'b6c0b50a-d0e3-43bf-a2a4-8d4674c2a7e8',
 'age': 40,
 'email': 'ih@ri.ro',
 'firstname': 'Emilie',
 'interests': ['Bowling', 'Cooking', 'Golf', 'Swimming'],
 'lastname': 'Woods',
 'workplace_id': 'a32bf18d-e0e5-48f2-a851-aa49c80f9460'}


#### Question 2  (10 Marks)

Insert a new Employee with the following properties:

* First Name: Jake 
* Last Name: Sample
* Email: jakesample@email.com
* Age: 26
* Interest: Biking, Hiking

Also, this employee works for 'Union Planters Corp' and lives at '573 Wojhas Square, Victoria'.
Verify that the insert succeeded and display the generated employees _id attribute.

__HINT__ An Employee document references a Workplace and Address document

In [81]:
# --- Question 2 Implementation ---

# Workplace ID for 'Union Planters Corp'
workplace = db.workplace.find_one({"name": "Union Planters Corp"})

# Address ID for '573 Wojhas Square, Victoria'
# Note: We match the address and city fields to ensure accuracy
address = db.address.find_one({
    "address": "573 Wojhas Square", 
    "city": "Victoria"
})

if workplace and address:
    #  Construct the new employee document
    new_employee = {
        "firstname": "Jake",
        "lastname": "Sample",
        "email": "jakesample@email.com",
        "age": 26,
        "interests": ["Biking", "Hiking"],
        "workplace_id": workplace["_id"],  # Reference to Workplace
        "address_id": address["_id"]       # Reference to Address
    }

    # Insert the new document
    insert_result = db.employee.insert_one(new_employee)

    # Verification
    print("Success: New employee inserted.")
    print(f"Generated Employee _id: {insert_result.inserted_id}")

else:
    print("Error: Could not find the specified Workplace or Address in the database.")

Success: New employee inserted.
Generated Employee _id: 69cc6ea7df8a9f74e981125e


#### Question 3 (10 Marks)

Delete all employees that work for 'Great Plains Energy Inc.' and are greater than 46 years old and likes 'Tennis'. Once you delete the employees verify the number of employees deleted.

In [82]:
# --- Question 3 Implementation ---

# Find the workplace ID for 'Great Plains Energy Inc.'
workplace_doc = db.workplace.find_one({"name": "Great Plains Energy Inc."})

if workplace_doc:
    target_workplace_id = workplace_doc["_id"]
    
    # Define the deletion criteria
    # We look for the specific workplace, age > 46, and 'Tennis' in the interests array
    delete_filter = {
        "workplace_id": target_workplace_id,
        "age": {"$gt": 46},
        "interests": "Tennis"
    }
    
    # Execute the deletion
    delete_result = db.employee.delete_many(delete_filter)
    
    # Verification
    print(f"Deletion Summary:")
    print(f"- Workplace Name: Great Plains Energy Inc.")
    print(f"- Criteria: Age > 46, Interests: Tennis")
    print(f"- Total Employees Deleted: {delete_result.deleted_count}")
    
else:
    print("Error: 'Great Plains Energy Inc.' not found in the Workplace collection.")

Deletion Summary:
- Workplace Name: Great Plains Energy Inc.
- Criteria: Age > 46, Interests: Tennis
- Total Employees Deleted: 4


#### Question 4 (12 Marks)
Add a new field called 'industry' to all employees that work for 'Health Net Inc.' and populate the field with the value 'Health Care'.

__HINT__ Adding a new field to a document is like updating the document

In [83]:
# --- Question 4 Implementation ---

# Find the workplace ID for 'Health Net Inc.'
workplace_doc = db.workplace.find_one({"name": "Health Net Inc."})

if workplace_doc:
    target_id = workplace_doc["_id"]
    
    # Update all matching employees to add the 'industry' field
    result = db.employee.update_many(
        {"workplace_id": target_id},
        {"$set": {"industry": "Health Care"}}
    )
    
    print(f"Update Results:")
    print(f"- Employees matched: {result.matched_count}")
    print(f"- Employees updated: {result.modified_count}")
    
    # 3. Clean Output: Display the result in a Table (DataFrame)
    # We query the updated employees to verify the new field exists
    updated_list = list(db.employee.find({"workplace_id": target_id}))
    df = pd.DataFrame(updated_list)
    
    # Displaying first 5 rows with relevant columns for the assignment
    print("\nVerification Table:")
    display(df[['firstname', 'lastname', 'industry']].head())
    
else:
    print("Error: 'Health Net Inc.' not found in the Workplace collection.")

Update Results:
- Employees matched: 14
- Employees updated: 14

Verification Table:


,firstname,lastname,industry
0,Marc,Castro,Health Care
1,Lucinda,Gomez,Health Care
2,Emilie,Robertson,Health Care
3,Josephine,Mills,Health Care
4,Vernon,Edwards,Health Care


#### Question 5 (10 Marks)

Create an aggregate query to count the number of employees for each company and sort the output from largest employee count to lowest employee count.

__NOTE__ you will use a pipeline to achieve the computed result.  You should produce a result similar to the following table (the following table contains fake data)
<table>
    <tr><th></th><th>_id</th><th>count</th></tr>
    <tr><td>0</td><td>[Equity Residential Properties Trust]</td><td>19</td></tr>
    <tr><td>...</td><td>...</td><td>...</td></tr>
    <tr><td>7</td><td>[Bell Microproducts Inc.]</td><td>6</td></tr>
    <tr><td>8</td><td>[Kemet Corp.]</td><td>1</td></tr>
</table>

__HINT__ you should make use of the \\$lookup, \\$group and \\$sort pipeline operations

In [84]:
# YOUR CODE HERE

query_result = db.employee.aggregate([
    
    {
        "$lookup": {
            "from": "workplace",
            "localField": "workplace_id",
            "foreignField": "_id",
            "as": "workplace_info"
        }
    },
    
    {"$unwind": "$workplace_info"},
    
    {
        "$group": {
            "_id": "$workplace_info.name",
            "count": {"$sum": 1}
        }
    },
    
    {"$sort": {"count": -1}}
])

pd.DataFrame(list(query_result))

,_id,count
0,Hilton Solutions,15
1,Health Net Inc.,14
2,Aetna Inc.,13
3,Bell Microproducts Inc.,11
4,Union Planters Corp,10
5,Equity Office Properties Trust,10
6,Equity Residential Properties Trust,7
7,Kemet Corp.,6
8,Xcel Bear Inc,6
9,Great Plains Energy Inc.,5


Analysis: MongoDB Data Management
In this assignment, I built a local data management system using MongoDB and PyMongo. My goal was to transform raw JSON files into a functional database and then perform specific administrative tasks like data insertion, multi-condition deletions, and schema updates.

1. System Architecture and Data Loading
I started by establishing a connection between my Jupyter Notebook and a local MongoDB server using MongoClient. I treated the provided JSON files (Employee, Workplace, and Address) as my data source. I developed a Python function to automate the "Extract and Load" process, which parsed these files and populated three distinct collections in my database.

2. Handling Document Relationships
A key part of my analysis was managing Document Referencing. Unlike traditional SQL databases that use JOINs, I handled relationships manually by looking up _id values. For example, when I needed to add "Jake Sample" to the database, I couldn't just type his workplace name. I first had to query the workplace collection to find the unique ID for "Union Planters Corp" and then reference that ID inside the new Employee document to maintain data integrity.

3. Executing CRUD Operations
I applied various MongoDB operators to fulfill the assignment requirements:

Querying: I used the $lte (less than or equal to) operator to filter employees by age and demonstrated how MongoDB can natively search for a specific value inside an array of "Interests."

Deletion: I performed a targeted cleanup by combining multiple filters—Workplace ID, Age, and Interests—to ensure only specific records were removed using delete_many.

Schema Evolution: I demonstrated the flexibility of NoSQL by adding a new industry field to existing documents. By using the $set operator, I was able to update a specific subset of employees without affecting the rest of the document structure.

4. Conclusion
I successfully moved from static data storage to a dynamic environment where I could query and manipulate complex relationships. This workflow mirrors how data is handled in real-world Big Data environments, where scalability and flexible schemas are essential for managing modern datasets.